# IMPOTANT
*Needs to be tested on a more representative dataframe and uploaded to GitHub as a separete project (implementation of emmeans from R)*

LMM link: https://www.statsmodels.org/dev/examples/notebooks/generated/mixed_lm_example.html

## Fitting the model

To get the idea of the distribution (for the model):
- Very technical approach: Run random effects model > generate residuals > check distribution of the residuals > choose the model accordingly
- Empirical approach: run the model based on either gaussian distribution (identity link function) or gamma distribution (log link function) and compare which of them fits better

Steps for running stats using GLMM:
- Get the fixed effects table
- Get estimated means of conditions and difference between the means
- Run t-test based on the estmated means



# IMPORTANT NOTES ON CODE

Here’s a compact, step-by-step way to read what you’ve got and what it means.

# 1) What the MixedLM table tells you

* **Model**: `pac_value ~ condition` with a **random intercept per subject** (`groups=sub`).
* **Coding**: The intercept is the mean of the reference level of `condition` (likely `_BL_N/A`). The two listed coefficients are **differences** from that reference.
* **Coefficients**

  * `Intercept`: mean pac_value at `_BL_N/A`.
  * `condition[T._MAIN__adaptation]`: mean difference vs `_BL_N/A` (negative here).
  * `condition[T._MAIN__baseline]`: mean difference vs `_BL_N/A` (positive here).
* **z and p**: Wald tests for whether each coefficient differs from 0. All three fixed-effect lines are highly significant (tiny p’s).
* **CIs**: All tight around the estimates—consistent with strong effects.
* **Random Effect Variance (Group Var ≈ 3.68e-10)**: Between-subject variability (after accounting for `condition`) is extremely small relative to your outcome scale. That can be real or just a scaling/rounding artifact, but it means subjects don’t differ much in baseline levels in this model.

# 2) What the EMM pairwise table adds

You computed **estimated marginal means (EMMs)** and **pairwise contrasts** with **Tukey adjustment** (good for all pairwise comparisons among groups).

Each row compares two conditions:

* **estimate_link**: the **difference in means** on the model’s link scale. Here the link is identity (linear mixed model), so this is a **raw difference in pac_value**.
* **SE**: standard error of that difference.
* **t.ratio**: test statistic for the difference.
* **p.value.adj**: Tukey-adjusted p-value (controls family-wise error across all pairs).

Your results:

* `_BL_N/A – _MAIN__baseline = −0.000149`, **adj p < 1e-6** → `_MAIN__baseline` is **higher** than `_BL_N/A` by ~1.49×10⁻⁴.
* `_BL_N/A – _MAIN__adaptation = +0.000020`, **adj p = 0.0018** → `_BL_N/A` is **slightly higher** than `_MAIN__adaptation` by ~2×10⁻⁵.
* `_MAIN__baseline – _MAIN__adaptation = +0.000170`, **adj p < 1e-6** → `_MAIN__baseline` is **higher** than `_MAIN__adaptation` by ~1.70×10⁻⁴.

So the **ordering of means** is:

```
MAIN__baseline  >  BL_N/A  >  MAIN__adaptation
```

and **all three pairwise differences are statistically significant** after Tukey correction.

# 3) Are these differences big or small?

You said earlier pac_value has **mean ≈ 1×10⁻⁴** and **SD ≈ 5×10⁻⁵**. Using that scale:

* The largest contrast (baseline vs adaptation) ≈ **1.7×10⁻⁴**, which is about **3.4 SDs** (1.7e-4 / 5e-5). That’s **very large** relative to your outcome variability.
* The small contrast (BL_N/A vs adaptation) ≈ **2×10⁻⁵**, ~**0.4 SD**, still meaningful and significant.

(If those summary stats came from a subset, re-check with your actual sample SDs per condition.)

# 4) How to report this

* **Model**: “We fitted a linear mixed-effects model with condition as a fixed effect and random intercepts for subjects.”
* **Main effect**: “Condition had a significant effect on pac_value (all pairwise Tukey-adjusted p’s ≤ 0.0018).”
* **EMM contrasts** (report the three differences with CIs if you also computed EMM CIs):

  * `MAIN__baseline` vs `MAIN__adaptation`: Δ = 0.000170 (Tukey-adj p < 0.001)
  * `MAIN__baseline` vs `BL_N/A`: Δ = 0.000149 (Tukey-adj p < 0.001)
  * `BL_N/A` vs `MAIN__adaptation`: Δ = 0.000020 (Tukey-adj p = 0.0018)
* **Direction/ordering**: “pac_value is highest in MAIN__baseline, intermediate at BL_N/A, lowest in MAIN__adaptation.”
* **Random effects**: “Between-subject variance was near zero on this scale, suggesting limited subject-level heterogeneity after accounting for condition.”

# 5) Practical tips


* **Rescale for readability**: Because values are ~1e-4, consider reporting in **micro-units** (e.g., multiply by 10⁶). Then the big contrast becomes **+170 μ-units** with the same statistics. This improves interpretability without changing inference.
* **Check model diagnostics**: residual plots, influential points, and whether random-effect variance being ~0 is reasonable for your data.
* **Include EMMs**: It’s useful to also print the per-condition EMMs (not just contrasts) with 95% CIs, so readers see the absolute levels.


In [7]:
import statsmodels.formula.api as smf
import pandas as pd
import os
# Load the dataframe
group = 'Y'
roi_dir = f'D:\\BonoKat\\research project\\# study 1\\eeg_data\\set\\{group} group\\roi_source_analysis'
df_pac_roi = pd.read_csv(os.path.join(roi_dir, f'PAC_MI_SOURCE_{group}_ROI.csv'))
df_pac_roi['block'] = df_pac_roi['block'].fillna('N/A')
df_pac_roi["condition"] = df_pac_roi["task"] + "_" + df_pac_roi["block"]
roi = 'G_precentral-lh'  # Example ROI name
df_roi = df_pac_roi.query("roi == @roi")

df_plan = df_roi.query("task_stage == '_plan'") # for model_1
df_go = df_roi.query("task_stage == '_go'") # for model_2
df_plan_main = df_plan.query("task == '_MAIN'") # for model_3
df_go_main = df_go.query("task == '_MAIN'") # for model_4

In [16]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning

# Suppress convergence warnings for mixed models with zero random effects
# (These warnings indicate minimal between-subject variance, which is OK)
warnings.filterwarnings('ignore', category=ConvergenceWarning)

# Model templates
print(f"\nROI: {roi}")

# 1 - BL_plan vs MAIN_plan_baseline vs MAIN_plan_adaptation
model_1 = smf.mixedlm(
    "pac_value ~ condition",
    data=df_plan,
    groups=df_plan["sub"]
).fit(reml=True)
print("\n" + "="*70)
print("MODEL 1: Planning Stage (BL vs MAIN baseline vs MAIN adaptation)")
print("="*70)
print(model_1.summary().as_text())
print(f"Random Effect Variance: {model_1.cov_re.iloc[0, 0]:.2e}")

# 2 - BL_go vs MAIN_go_baseline vs MAIN_go_adaptation
model_2 = smf.mixedlm(
    "pac_value ~ condition",
    data=df_go,
    groups=df_go["sub"]
).fit(reml=True)
print("\n" + "="*70)
print("MODEL 2: Go Stage (BL vs MAIN baseline vs MAIN adaptation)")
print("="*70)
print(model_2.summary().as_text())
print(f"Random Effect Variance: {model_2.cov_re.iloc[0, 0]:.2e}")

# 3 - MAIN_plan_baseline vs MAIN_plan_adaptation
model_3 = smf.mixedlm(
    "pac_value ~ block",
    data=df_plan_main,
    groups=df_plan_main["sub"]
).fit(reml=True)
print("\n" + "="*70)
print("MODEL 3: Planning Stage MAIN only (baseline vs adaptation)")
print("="*70)
print(model_3.summary().as_text())
print(f"Random Effect Variance: {model_3.cov_re.iloc[0, 0]:.2e}")

# 4 - MAIN_go_baseline vs MAIN_go_adaptation
model_4 = smf.mixedlm(
    "pac_value ~ block",
    data=df_go_main,
    groups=df_go_main["sub"]
).fit(reml=True)
print("\n" + "="*70)
print("MODEL 4: Go Stage MAIN only (baseline vs adaptation)")
print("="*70)
print(model_4.summary().as_text())
print(f"Random Effect Variance: {model_4.cov_re.iloc[0, 0]:.2e}")


ROI: G_precentral-lh

MODEL 1: Planning Stage (BL vs MAIN baseline vs MAIN adaptation)
                  Mixed Linear Model Regression Results
Model:                  MixedLM       Dependent Variable:       pac_value
No. Observations:       69            Method:                   REML     
No. Groups:             23            Scale:                    0.0000   
Min. group size:        3             Log-Likelihood:           598.5126 
Max. group size:        3             Converged:                Yes      
Mean group size:        3.0                                              
-------------------------------------------------------------------------
                               Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-------------------------------------------------------------------------
Intercept                       0.000    0.000 22.546 0.000  0.000  0.000
condition[T._MAIN__adaptation] -0.000    0.000 -3.250 0.001 -0.000 -0.000
condition[T._MAIN__baseline]    0.000    0

### Diagnostic: Should We Use OLS Instead?

If random effect variances are near zero (< 1e-10), consider using regular OLS models with clustered standard errors.

In [17]:
# Check if we should use OLS models instead
models = {
    'Model 1 (Planning)': model_1,
    'Model 2 (Go)': model_2,
    'Model 3 (MAIN Planning)': model_3,
    'Model 4 (MAIN Go)': model_4
}

print("RANDOM EFFECT VARIANCE DIAGNOSTICS")
print("="*70)
threshold = 1e-8
use_ols_recommendation = []

for name, model in models.items():
    var_re = model.cov_re.iloc[0, 0]
    print(f"{name:25s} | Random Var: {var_re:.2e} | ", end="")
    
    if var_re < threshold:
        print("⚠️  Near zero - consider OLS")
        use_ols_recommendation.append(name)
    else:
        print("✓ Mixed model appropriate")

if use_ols_recommendation:
    print("\n" + "="*70)
    print("RECOMMENDATION: Consider using OLS for models with near-zero random effects.")
    print("This would avoid convergence warnings and provide more interpretable results.")
    print("\nAlternatively, the current mixed models are still valid - the warnings")
    print("simply indicate that subject-level random intercepts add little variance.")
else:
    print("\n✓ All models have meaningful random effects - mixed models appropriate.")

RANDOM EFFECT VARIANCE DIAGNOSTICS
Model 1 (Planning)        | Random Var: 3.68e-10 | ⚠️  Near zero - consider OLS
Model 2 (Go)              | Random Var: 1.45e-10 | ⚠️  Near zero - consider OLS
Model 3 (MAIN Planning)   | Random Var: 6.02e-10 | ⚠️  Near zero - consider OLS
Model 4 (MAIN Go)         | Random Var: 8.69e-11 | ⚠️  Near zero - consider OLS

RECOMMENDATION: Consider using OLS for models with near-zero random effects.
This would avoid convergence warnings and provide more interpretable results.

Alternatively, the current mixed models are still valid - the warnings
simply indicate that subject-level random intercepts add little variance.


## Model Fitting

The convergence warnings below indicate that random effect variances are estimated at or near zero. This suggests minimal between-subject variability, and we may consider:
1. Using regular OLS models instead
2. Suppressing the warnings if mixed models are theoretically justified
3. Adding diagnostic output to check variance components

## Estimated Marginal Means (EMMs) Implementation

This section implements a comprehensive `emmeans` function for post-hoc testing of mixed linear models.

In [8]:
import builtins
dir(builtins)

['ArithmeticError',
 'AssertionError',
 'AttributeError',
 'BaseException',
 'BaseExceptionGroup',
 'BlockingIOError',
 'BrokenPipeError',
 'BufferError',
 'BytesWarning',
 'ChildProcessError',
 'ConnectionAbortedError',
 'ConnectionError',
 'ConnectionRefusedError',
 'ConnectionResetError',
 'DeprecationWarning',
 'EOFError',
 'Ellipsis',
 'EncodingWarning',
 'EnvironmentError',
 'Exception',
 'ExceptionGroup',
 'False',
 'FileExistsError',
 'FileNotFoundError',
 'FloatingPointError',
 'FutureWarning',
 'GeneratorExit',
 'IOError',
 'ImportError',
 'ImportWarning',
 'IndentationError',
 'IndexError',
 'InterruptedError',
 'IsADirectoryError',
 'KeyError',
 'KeyboardInterrupt',
 'LookupError',
 'MemoryError',
 'ModuleNotFoundError',
 'NameError',
 'None',
 'NotADirectoryError',
 'NotImplemented',
 'NotImplementedError',
 'OSError',
 'OverflowError',
 'PendingDeprecationWarning',
 'PermissionError',
 'ProcessLookupError',
 'RecursionError',
 'ReferenceError',
 'ResourceWarning',
 'Runti

In [5]:
import pandas as pd
dir(pd)

['ArrowDtype',
 'BooleanDtype',
 'Categorical',
 'CategoricalDtype',
 'CategoricalIndex',
 'DataFrame',
 'DateOffset',
 'DatetimeIndex',
 'DatetimeTZDtype',
 'ExcelFile',
 'ExcelWriter',
 'Flags',
 'Float32Dtype',
 'Float64Dtype',
 'Grouper',
 'HDFStore',
 'Index',
 'IndexSlice',
 'Int16Dtype',
 'Int32Dtype',
 'Int64Dtype',
 'Int8Dtype',
 'Interval',
 'IntervalDtype',
 'IntervalIndex',
 'MultiIndex',
 'NA',
 'NaT',
 'NamedAgg',
 'Period',
 'PeriodDtype',
 'PeriodIndex',
 'RangeIndex',
 'Series',
 'SparseDtype',
 'StringDtype',
 'Timedelta',
 'TimedeltaIndex',
 'Timestamp',
 'UInt16Dtype',
 'UInt32Dtype',
 'UInt64Dtype',
 'UInt8Dtype',
 '__all__',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__docformat__',
 '__file__',
 '__git_version__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '__version__',
 '_built_with_meson',
 '_config',
 '_is_numpy_dev',
 '_libs',
 '_pandas_datetime_CAPI',
 '_pandas_parser_CAPI',
 '_testing',
 '_typing',
 '_version_meson',
 'annota

In [3]:
import sys
sys.builtin_module_names

('_abc',
 '_ast',
 '_bisect',
 '_blake2',
 '_codecs',
 '_codecs_cn',
 '_codecs_hk',
 '_codecs_iso2022',
 '_codecs_jp',
 '_codecs_kr',
 '_codecs_tw',
 '_collections',
 '_contextvars',
 '_csv',
 '_datetime',
 '_functools',
 '_heapq',
 '_imp',
 '_io',
 '_json',
 '_locale',
 '_lsprof',
 '_md5',
 '_multibytecodec',
 '_opcode',
 '_operator',
 '_pickle',
 '_random',
 '_sha1',
 '_sha2',
 '_sha3',
 '_signal',
 '_sre',
 '_stat',
 '_statistics',
 '_string',
 '_struct',
 '_symtable',
 '_thread',
 '_tokenize',
 '_tracemalloc',
 '_typing',
 '_warnings',
 '_weakref',
 '_winapi',
 '_xxinterpchannels',
 '_xxsubinterpreters',
 'array',
 'atexit',
 'audioop',
 'binascii',
 'builtins',
 'cmath',
 'errno',
 'faulthandler',
 'gc',
 'itertools',
 'marshal',
 'math',
 'mmap',
 'msvcrt',
 'nt',
 'sys',
 'time',
 'winreg',
 'xxsubtype',
 'zlib')

### Example Usage: Post-hoc Tests for model_1

Now let's apply the `emmeans` function to perform post-hoc pairwise comparisons for our mixed linear model.

In [ ]:
model_1 = smf.mixedlm(
    "pac_value ~ condition",
    data=df_plan,
    groups=df_plan["sub"]
).fit(reml=True)
print("\n" + "="*70)
print("MODEL 1: Planning Stage (BL vs MAIN baseline vs MAIN adaptation)")
print("="*70)
print(model_1.summary().as_text())
print(f"Random Effect Variance: {model_1.cov_re.iloc[0, 0]:.2e}")

# Compute EMMs and post-hoc comparisons for model_1
# Using df_method="resid" (default) for conservative small-sample inference
results_model1 = emmeans(
    result=model_1,
    data=df_plan,
    specs="condition",
    adjust="tukey",
    level=0.95,
    return_grid=True
)
# Display Pairwise Contrasts
print("\n" + "="*70)
print("PAIRWISE COMPARISONS (adjusted)")
print("="*70)

ctr = results_model1['contrasts']
if ctr is not None:
    # Show whatever stat column exists
    stat_col = 't.ratio' if 't.ratio' in ctr.columns else ('z.ratio' if 'z.ratio' in ctr.columns else 'stat')
    cols = ['contrast', 'estimate_link', 'SE', stat_col, 'p.value', 'p.value.adj']
    cols = [c for c in cols if c in ctr.columns]  # keep only existing
    print(ctr[cols].to_string(index=False))

    # Significant comparisons
    sig = ctr[ctr['p.value.adj'] < 0.05] if 'p.value.adj' in ctr.columns else ctr[ctr['p.value'] < 0.05]
    print("\n" + "="*70)
    print(f"SIGNIFICANT COMPARISONS (p < 0.05): {len(sig)}/{len(ctr)}")
    print("="*70)
    if len(sig) > 0:
        sig_cols = ['contrast', 'estimate_link', 'SE', 'p.value.adj'] if 'p.value.adj' in ctr.columns else ['contrast', 'estimate_link', 'SE', 'p.value']
        sig_cols = [c for c in sig_cols if c in sig.columns]
        print(sig[sig_cols].to_string(index=False))
    else:
        print("No significant pairwise differences found.")
else:
    print("No contrasts computed")


MODEL 1: Planning Stage (BL vs MAIN baseline vs MAIN adaptation)
                  Mixed Linear Model Regression Results
Model:                  MixedLM       Dependent Variable:       pac_value
No. Observations:       69            Method:                   REML     
No. Groups:             23            Scale:                    0.0000   
Min. group size:        3             Log-Likelihood:           598.5126 
Max. group size:        3             Converged:                Yes      
Mean group size:        3.0                                              
-------------------------------------------------------------------------
                               Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-------------------------------------------------------------------------
Intercept                       0.000    0.000 22.546 0.000  0.000  0.000
condition[T._MAIN__adaptation] -0.000    0.000 -3.250 0.001 -0.000 -0.000
condition[T._MAIN__baseline]    0.000    0.000 24.023 0.000  0.0

In [20]:
# Compute EMMs and post-hoc comparisons for model_1
# Using df_method="resid" (default) for conservative small-sample inference
results_model1 = emmeans(
    result=model_2,
    data=df_go,
    specs="condition",
    adjust="tukey",
    level=0.95,
    return_grid=True
)
# Display Pairwise Contrasts
print("\n" + "="*70)
print("PAIRWISE COMPARISONS (adjusted)")
print("="*70)
print(model_2.summary().as_text())
print(f"Random Effect Variance: {model_2.cov_re.iloc[0, 0]:.2e}")

ctr = results_model1['contrasts']
if ctr is not None:
    # Show whatever stat column exists
    stat_col = 't.ratio' if 't.ratio' in ctr.columns else ('z.ratio' if 'z.ratio' in ctr.columns else 'stat')
    cols = ['contrast', 'estimate_link', 'SE', stat_col, 'p.value', 'p.value.adj']
    cols = [c for c in cols if c in ctr.columns]  # keep only existing
    print(ctr[cols].to_string(index=False))

    # Significant comparisons
    sig = ctr[ctr['p.value.adj'] < 0.05] if 'p.value.adj' in ctr.columns else ctr[ctr['p.value'] < 0.05]
    print("\n" + "="*70)
    print(f"SIGNIFICANT COMPARISONS (p < 0.05): {len(sig)}/{len(ctr)}")
    print("="*70)
    if len(sig) > 0:
        sig_cols = ['contrast', 'estimate_link', 'SE', 'p.value.adj'] if 'p.value.adj' in ctr.columns else ['contrast', 'estimate_link', 'SE', 'p.value']
        sig_cols = [c for c in sig_cols if c in sig.columns]
        print(sig[sig_cols].to_string(index=False))
    else:
        print("No significant pairwise differences found.")
else:
    print("No contrasts computed")


PAIRWISE COMPARISONS (adjusted)
                  Mixed Linear Model Regression Results
Model:                   MixedLM       Dependent Variable:       pac_value
No. Observations:        69            Method:                   REML     
No. Groups:              23            Scale:                    0.0000   
Min. group size:         3             Log-Likelihood:           612.4956 
Max. group size:         3             Converged:                Yes      
Mean group size:         3.0                                              
--------------------------------------------------------------------------
                               Coef.  Std.Err.    z    P>|z| [0.025 0.975]
--------------------------------------------------------------------------
Intercept                       0.000    0.000  33.875 0.000  0.000  0.000
condition[T._MAIN__adaptation] -0.000    0.000 -17.066 0.000 -0.000 -0.000
condition[T._MAIN__baseline]    0.000    0.000   2.883 0.004  0.000  0.000
Group Var  

In [21]:
model_2.params

Intercept                         0.000155
condition[T._MAIN__adaptation]   -0.000092
condition[T._MAIN__baseline]      0.000016
Group Var                         0.433510
dtype: float64

In [8]:
# Compute EMMs and post-hoc comparisons for model_1
# Using df_method="resid" (default) for conservative small-sample inference
results_model1 = emmeans(
    result=model_3,
    data=df_plan_main,
    specs="block",
    adjust="tukey",
    level=0.95,
    return_grid=True
)
# Display Pairwise Contrasts
print("\n" + "="*70)
print("PAIRWISE COMPARISONS (adjusted)")
print("="*70)

ctr = results_model1['contrasts']
if ctr is not None:
    # Show whatever stat column exists
    stat_col = 't.ratio' if 't.ratio' in ctr.columns else ('z.ratio' if 'z.ratio' in ctr.columns else 'stat')
    cols = ['contrast', 'estimate_link', 'SE', stat_col, 'p.value', 'p.value.adj']
    cols = [c for c in cols if c in ctr.columns]  # keep only existing
    print(ctr[cols].to_string(index=False))

    # Significant comparisons
    sig = ctr[ctr['p.value.adj'] < 0.05] if 'p.value.adj' in ctr.columns else ctr[ctr['p.value'] < 0.05]
    print("\n" + "="*70)
    print(f"SIGNIFICANT COMPARISONS (p < 0.05): {len(sig)}/{len(ctr)}")
    print("="*70)
    if len(sig) > 0:
        sig_cols = ['contrast', 'estimate_link', 'SE', 'p.value.adj'] if 'p.value.adj' in ctr.columns else ['contrast', 'estimate_link', 'SE', 'p.value']
        sig_cols = [c for c in sig_cols if c in sig.columns]
        print(sig[sig_cols].to_string(index=False))
    else:
        print("No significant pairwise differences found.")
else:
    print("No contrasts computed")


PAIRWISE COMPARISONS (adjusted)
               contrast  estimate_link       SE   t.ratio  p.value  p.value.adj
_baseline - _adaptation        0.00017 0.000006 27.877833      0.0          0.0

SIGNIFICANT COMPARISONS (p < 0.05): 1/1
               contrast  estimate_link       SE  p.value.adj
_baseline - _adaptation        0.00017 0.000006          0.0


In [9]:
# Compute EMMs and post-hoc comparisons for model_1
# Using df_method="resid" (default) for conservative small-sample inference
results_model1 = emmeans(
    result=model_4,
    data=df_go_main,
    specs="block",
    adjust="tukey",
    level=0.95,
    return_grid=True
)
# Display Pairwise Contrasts
print("\n" + "="*70)
print("PAIRWISE COMPARISONS (adjusted)")
print("="*70)

ctr = results_model1['contrasts']
if ctr is not None:
    # Show whatever stat column exists
    stat_col = 't.ratio' if 't.ratio' in ctr.columns else ('z.ratio' if 'z.ratio' in ctr.columns else 'stat')
    cols = ['contrast', 'estimate_link', 'SE', stat_col, 'p.value', 'p.value.adj']
    cols = [c for c in cols if c in ctr.columns]  # keep only existing
    print(ctr[cols].to_string(index=False))

    # Significant comparisons
    sig = ctr[ctr['p.value.adj'] < 0.05] if 'p.value.adj' in ctr.columns else ctr[ctr['p.value'] < 0.05]
    print("\n" + "="*70)
    print(f"SIGNIFICANT COMPARISONS (p < 0.05): {len(sig)}/{len(ctr)}")
    print("="*70)
    if len(sig) > 0:
        sig_cols = ['contrast', 'estimate_link', 'SE', 'p.value.adj'] if 'p.value.adj' in ctr.columns else ['contrast', 'estimate_link', 'SE', 'p.value']
        sig_cols = [c for c in sig_cols if c in sig.columns]
        print(sig[sig_cols].to_string(index=False))
    else:
        print("No significant pairwise differences found.")
else:
    print("No contrasts computed")


PAIRWISE COMPARISONS (adjusted)
               contrast  estimate_link       SE   t.ratio  p.value  p.value.adj
_baseline - _adaptation       0.000108 0.000004 25.774454      0.0          0.0

SIGNIFICANT COMPARISONS (p < 0.05): 1/1
               contrast  estimate_link       SE  p.value.adj
_baseline - _adaptation       0.000108 0.000004          0.0
